# Approval Processing
Process a reviewed Excel workbook into versioned learned context and refresh SQLite.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
ROOT = Path.cwd()
if not (ROOT / 'config').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
from dq_agent.config import load_app_config
from dq_agent.context_store import read_context
from dq_agent.context_utils import configure_workflow_logging, context_workflow_paths, logged_step, process_approval_workbook, read_approval_workbook

In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
APPROVAL_FILE = None  # Set to a completed workbook under approvals/reviewed
config = load_app_config(ROOT)
paths = context_workflow_paths(config, RUN_ID)
logger = configure_workflow_logging(paths['log'], config.project.log_level)
display([str(path) for path in sorted(paths['reviewed'].glob('*.xlsx'))])

In [ ]:
review_frame = None
if APPROVAL_FILE is None:
    print('No approval workbook selected. Set APPROVAL_FILE after moving a completed workbook to approvals/reviewed.')
else:
    APPROVAL_FILE = Path(APPROVAL_FILE)
    review_frame = read_approval_workbook(APPROVAL_FILE)
    display(review_frame)

In [ ]:
if review_frame is not None:
    with logged_step(logger, paths['checkpoint'], 'PROCESS_APPROVAL', file=str(APPROVAL_FILE)):
        result = process_approval_workbook(config, APPROVAL_FILE)
    display(result)
    display(read_context(config).head(20))
else:
    print('Approval processing skipped.')

Approved records are written to context_layer/learned_context.yaml; processed workbooks move to approvals/processed and older versions to archive.